# 34. XGBoost's depth, which was never fitted to XGBoost

**One variable against ledger row 38** (`xgb_te`, CV 0.967099): `max_depth`. Same
learner, same folds, same seed, same budget, same encoder, same everything else.

## Why this is not a fourth null sweep

Three hyperparameter sweeps have returned nothing here: `num_leaves`, `SMOOTH` and
`N_INNER`. The rule those support says a change of representation revalues learners and
not their knobs, and on that basis modelling was closed.

This one is different in a way worth stating precisely, because if the distinction is
wrong then the rule predicts a null and the run is a waste.

`num_leaves` was **LightGBM's own default, on LightGBM**. `SMOOTH` and `N_INNER` were
the encoder's own constants, on the encoder. Each was a knob sitting at a value chosen
for the component it belongs to, and each turned out to be near-optimal, which is what
the rule predicts for a knob already fitted to its representation.

`max_depth = 6` is not that. It was **inherited from the LightGBM budget convention** on
2026-08-19 when `24_xgboost_te.ipynb` was written, alongside `lr = 0.05` and
`n_estimators = 2000`, so XGBoost would be compared to LightGBM at a matched budget.
That was the correct choice for the comparison it was making. It has never been fitted
to XGBoost, and XGBoost's default is 6 only because that is the library's default, which
is not evidence about this dataset with 691,369 rows.

So the honest framing: this is a knob that has never been fitted to **anything**, and it
is the only one left in the repo. The rule does not obviously cover it.

## The prior, and it is not symmetric

Depth controls the order of interaction a tree can represent. At 691,369 rows there is
enough data to support deeper trees than 6 without the usual overfitting penalty, and
`32` gave a specific reason to care: the crossed encodings were worth nothing precisely
**because the trees already reach 2-way regions**. Whether they reach 3-way and 4-way
regions is a question about depth and nothing else.

Against that: `n_estimators` is fixed at 2000 with `lr = 0.05`, so a deeper model is
also a much larger model on a fixed budget, and the turnover seen at 2000 trees in
experiment 5 says this data punishes capacity eventually.

**Prediction, before the run: a small gain at depth 8, turning over by depth 10.** Last
prediction was wrong, in `32`, so this one is worth no more than the arithmetic behind
it.

## Cost, which is the reason the grid stops at 10

Depth doubles the leaf count per level, so the arms are not equal cost. Depth 6 runs
about 2m 25s per fold at 36 features; 8 and 10 should be roughly 2x and 4x that. The
grid is 4, 6, 8, 10 rather than reaching 12 because 12 alone would cost more than the
other four arms together and the turnover, if there is one, will already be visible.

## The gate, pre-registered

Reused from `29`, `30` and `32` unchanged, judged against the `6` arm from this same
kernel.

| verdict | condition |
|---|---|
| `carry to a second seed` | wins >= 4/5, mean > 2 x paired sd, and mean >= +0.00010 |
| `parity` | paired-significant but under +0.00010 |
| `null` | anything else |


In [ ]:
# One flag. The sweep always runs top to bottom on Kaggle.
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# The knob under test. 6 is row 38's value, inherited from the LightGBM budget
# convention rather than chosen for XGBoost, so that arm is the reproduction check.
DEPTHS = [4, 6, 8, 10]
BASE_DEPTH = 6
MAX_DEPTH = BASE_DEPTH   # rebound per arm below

# Row 38's configuration, held.
LR = 0.05
N_EST = 2000
BENCH_EST = 200
PROBE_FOLD = 0
N_JOBS = -1

BASELINE_NAME = "xgb_te"
BASELINE_CV = 0.967099
EXPECTED_FOLD_SHA = "ec282b0968059676"

EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

GATE_FLOOR = 1.0e-04

print(f"SMOKE = {SMOKE}   depths {DEPTHS}   base {BASE_DEPTH}")


## Stage 1. Data, folds, leak checklist

The fold checksum is the only thing standing between an out-of-fold vector that
blends and one that is silently misaligned, so it is checked before anything trains
rather than after.

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

# Runs here or on Kaggle. Both are found by name rather than by assuming a shape.
KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
SUB = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "submissions"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

# Leak checklist, re-run rather than ticked by inspection. `id` is a contiguous row
# index that separates train from test perfectly, so it is a guaranteed leak if it
# ever reaches the model.
checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# ROW_IDX maps this run's rows back into the saved member vectors.
if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    # Measured 2026-08-19 and written up: at 16,000 rows this machine
    # runs 101x slower at n_jobs=-1 than at n_jobs=1, monotone in the thread count.
    # N_JOBS above is chosen to match row 17 on Kaggle at 691,369 rows, where it is
    # right. A smoke run produces no ledger number, so overriding it here costs
    # nothing and is the difference between two minutes and giving up on the check.
    N_JOBS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

## Stage 2. The encoder

Copied from `13_target_encoding.ipynb` so the feature set is row 17's feature set. A
copy is a provenance risk under the notebook layout, so it is checked rather than
asserted: the cell below parses the encoder out of `13`, normalises both versions
through `ast.unparse`, and compares checksums. Expected fingerprint
`0642e41750ef8bab`, the same value rows 26 and 33 recorded.

In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

### The leak checks, by execution

The same three checks `13` ran, on the same encoder, so their numbers are directly
comparable to the ones recorded earlier. Read all three together: the first two must be
about zero, the third must be large. Without the third, an encoder that ignored the
target entirely would pass the first two and look clean.

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## Stage 3. Bench and determinism

No liveness check is needed this time and it is worth saying why rather than quietly
dropping a stage that `29` and `30` both had. Those two swept constants read from module
globals by a function that could in principle have ignored them, so "did the knob reach
the code" was a real question. `max_depth` is a constructor argument to
`XGBClassifier`, and an arm that failed to pass it would raise rather than silently
produce a copy. The tree counts printed below make the same point positively: if depth
were not reaching the model, every arm would report the same model size.


In [ ]:
import xgboost as xgb


def make(n_est, depth):
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True,
        learning_rate=LR, n_estimators=n_est, max_depth=depth,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=N_JOBS, verbosity=0,
    )


def fit_arm(Xtr, ytr, Xva, n_est, depth, Xte=None):
    m = make(n_est, depth)
    t0 = time.time()
    m.fit(Xtr, ytr)
    secs = time.time() - t0
    p = m.predict_proba(Xva)[:, 1]
    p_te = m.predict_proba(Xte)[:, 1] if Xte is not None else None
    return p, p_te, secs, m


def hhmm(s):
    return f"{int(s // 60)}m {int(s % 60):02d}s"


LOG = (Path("/kaggle/working") if ON_KAGGLE
       else LOCAL / "artifacts" / "logs") / "34_xgb_depth.log"
LOG.parent.mkdir(parents=True, exist_ok=True)


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, depths={DEPTHS} ===")

tr0 = np.where(folds != PROBE_FOLD)[0]
va0 = np.where(folds == PROBE_FOLD)[0]
_t0 = time.time()
Xtr0, Xva0, _ = build(X, y, tr0, va0)
ENC_SECS = time.time() - _t0
print(f"encoder, one fold: {hhmm(ENC_SECS)}   {Xtr0.shape[1]} features")

pa, _, sa, ma = fit_arm(Xtr0, y[tr0], Xva0, BENCH_EST, BASE_DEPTH)
pb, _, _, _ = fit_arm(Xtr0, y[tr0], Xva0, BENCH_EST, BASE_DEPTH)
delta = float(np.abs(pa - pb).max())
DETERMINISTIC = delta == 0.0
print(f"determinism, two identical runs, max |diff|: {delta:.3e}  "
      f"{'OK' if DETERMINISTIC else 'NOT REPRODUCIBLE'}")

# Model size per arm, at the bench budget. This is the positive control: depth that
# failed to reach the model would print the same number four times.
print()
print("bench, and the leaf count is the check that depth is reaching the model:")
per_tree = {}
for d in DEPTHS:
    _, _, sd_, m = fit_arm(Xtr0, y[tr0], Xva0, BENCH_EST, d)
    leaves = sum(t.count("leaf") for t in m.get_booster().get_dump())
    per_tree[d] = sd_ / BENCH_EST
    print(f"  depth {d:>2}: {hhmm(sd_)} for {BENCH_EST} trees, {leaves:,} leaves total")

total = 5 * sum(ENC_SECS + per_tree[d] * N_EST for d in DEPTHS)
print()
print(f"projection, {N_EST} trees, 5 folds x {len(DEPTHS)} arms: {hhmm(total)}")
note(f"stage 3 done, determinism {'OK' if DETERMINISTIC else 'FAILED'}, "
     f"projected {hhmm(total)}")

del Xtr0, Xva0, pa, pb, ma
gc.collect()


## Stage 4. The sweep

The encoder is built once per fold and shared by all four arms, which is possible here
and was not in `29`: `SMOOTH` was an input to the encoder, `max_depth` is not.


In [ ]:
oof = {d: np.zeros(len(train)) for d in DEPTHS}
test_pred = {d: np.zeros(len(test)) for d in DEPTHS}
per_fold = {d: [] for d in DEPTHS}

t0 = time.time()
for f in range(5):
    tr = np.where(folds != f)[0]
    va = np.where(folds == f)[0]
    Xtr, Xva, Xte = build(X, y, tr, va, X_test)
    for d in DEPTHS:
        p, p_te, secs, _ = fit_arm(Xtr, y[tr], Xva, N_EST, d, Xte)
        oof[d][va] = p
        test_pred[d] += p_te / 5
        per_fold[d].append(float(roc_auc_score(y[va], p)))
        note(f"  fold {f} depth {d:>2}: {per_fold[d][-1]:.6f}  ({hhmm(secs)})")
        gc.collect()
    del Xtr, Xva, Xte
    gc.collect()
    done = time.time() - t0
    note(f"fold {f} done, elapsed {hhmm(done)}, "
         f"about {hhmm(done / (f + 1) * (4 - f))} left")

cv = {d: float(np.mean(per_fold[d])) for d in DEPTHS}
sd = {d: float(np.std(per_fold[d])) for d in DEPTHS}
print()
print(f"{'depth':>7} {'CV':>10} {'fold sd':>10}")
for d in DEPTHS:
    star = "  <- row 38's value, inherited from LightGBM" if d == BASE_DEPTH else ""
    print(f"{d:>7} {cv[d]:>10.6f} {sd[d]:>10.6f}{star}")
note("sweep done, " + ", ".join(f"{d}:{cv[d]:.6f}" for d in DEPTHS))


In [ ]:
repro = cv[BASE_DEPTH] - BASELINE_CV
REPRODUCED = abs(repro) < 1e-4
print(f"arm depth={BASE_DEPTH}: {cv[BASE_DEPTH]:.6f}")
print(f"ledger row 38   : {BASELINE_CV:.6f}")
print(f"difference      : {repro:+.2e}   "
      f"{'inside' if REPRODUCED else 'OUTSIDE'} the 1e-04 tolerance")
if SMOKE:
    print("SMOKE: subsampled, so this is EXPECTED to be far out and is not a check.")

base = np.array(per_fold[BASE_DEPTH])
rows = []
for d in DEPTHS:
    if d == BASE_DEPTH:
        continue
    diff = np.array(per_fold[d]) - base
    rows.append((d, diff.mean(), diff.std(ddof=1), int((diff > 0).sum()), diff))

print()
print(f"{'depth':>7} {'paired mean':>13} {'paired sd':>11} {'wins':>7} {'mean/sd':>9}")
for d, m, sdv, w, _ in rows:
    print(f"{d:>7} {m:>+13.6f} {sdv:>11.6f} {w:>5}/5 "
          f"{(m / sdv if sdv else float('nan')):>9.1f}")
print()
for d, m, sdv, w, diff in rows:
    print(f"  {d:>5}: per-fold {np.round(diff, 6).tolist()}")

order = [cv[d] for d in DEPTHS]
print()
print(f"CV in grid order {DEPTHS}: {[round(v, 6) for v in order]}")
peak = DEPTHS[int(np.argmax(order))]
print(f"peak at depth {peak}   "
      f"{'INTERIOR, so the grid brackets it' if peak not in (DEPTHS[0], DEPTHS[-1]) else 'AT A GRID EDGE, so the curve may continue'}")

blocked = None
if not (LEAK_OK and CLEAN):
    blocked = "a leak check failed"
elif not ENCODER_MATCH:
    blocked = "the encoder does not match 13"
elif not DETERMINISTIC:
    blocked = "the configuration is not reproducible"
elif not SMOKE and not ALIGNED:
    blocked = "fold alignment failed"
elif not SMOKE and not REPRODUCED:
    blocked = f"the depth={BASE_DEPTH} arm missed row 38 by {repro:+.2e}"

print()
if blocked:
    print(f"VERDICT: blocked, {blocked}")
elif SMOKE:
    print("SMOKE: no verdict, subsampled rows cannot resolve differences this small.")
else:
    d, m, sdv, w, _ = max(rows, key=lambda r: r[1])
    sig = w >= 4 and sdv > 0 and m > 2 * sdv
    if sig and m >= GATE_FLOOR:
        print(f"VERDICT: carry to a second seed. depth={d} clears the bar at {m:+.6f},")
        print("  and one seed does not make an improvement in this repo (rows 26, 38).")
    elif sig:
        print(f"VERDICT: parity. depth={d} is paired-significant at {m:+.6f} but under")
        print(f"  the {GATE_FLOOR:.5f} floor. Logged as parity, not as an improvement.")
    else:
        print("VERDICT: null. Depth 6 stands, and the knob that had never been fitted")
        print("  to anything was already right, which is the fourth such result here.")


In [ ]:
pre = "SMOKE_" if SMOKE else ""
for d in DEPTHS:
    np.save(OUT / f"{pre}xgb_depth{d}_oof.npy", oof[d])
    np.save(OUT / f"{pre}xgb_depth{d}_test.npy", test_pred[d])
print(f"wrote {pre}xgb_depth<d>_oof.npy and _test.npy for {DEPTHS}")
print()
print("ledger lines, one per arm:")
for d in DEPTHS:
    print(f"  xgb_depth{str(d):<3}  cv_mean {cv[d]:.6f}  cv_std {sd[d]:.6f}")
print()
print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}, "
      f"encoder {'matches 13' if ENCODER_MATCH else 'DIFFERS'}, "
      f"determinism {'OK' if DETERMINISTIC else 'FAILED'}")
